In [ ]:
!pip uninstall -y bitsandbytes triton
!pip install --no-cache-dir --upgrade "bitsandbytes>=0.45.5" "accelerate" "transformers" "peft" "trl"


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


In [ ]:
import gc
import glob as _glob
import os
import sys
import json
import re
import ast
import random
from numbers import Integral, Real

import pandas as pd
import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)
from IPython.display import display

DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

ADAPTER_DIR_P2 = f"{DRIVE_DIR}/mistral-react-adapter_divyam_duplicate"
CKPT_DIR_P2 = f"{DRIVE_DIR}/mistral-react-qlora_divyam_duplicate"

ADAPTER_DIR_DPO = f"{DRIVE_DIR}/mistral-react-dpo-adapter_divyam_duplicate3"
CKPT_DIR_DPO = f"{DRIVE_DIR}/mistral-react-dpo-ckpts_divyam_duplicate3"

MAX_SEQ_LENGTH_P2 = 1024
MAX_STEPS_INFER = 5
MAX_NEW_TOKENS_INFER = 256

sys.path.insert(0, DRIVE_DIR)
from tool_executor import ToolExecutor

df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")
with open(f"{DRIVE_DIR}/agent_trajectories_2k.json", "r", encoding="utf-8") as f:
    trajectories = json.load(f)

print("data shape:", df.shape)
print("trajectories:", len(trajectories))


def latest_path(paths, key_fn):
    if not paths:
        return None
    return sorted(paths, key=key_fn)[-1]


def has_adapter(path):
    return os.path.isfile(f"{path}/adapter_config.json")


def resolve_sft_load_path():
    epoch_ckpts = _glob.glob(f"{CKPT_DIR_P2}/checkpoint-*")
    timed_ckpts = _glob.glob(f"{ADAPTER_DIR_P2}/timed_ckpt_step_*")
    final_adapter_exists = has_adapter(ADAPTER_DIR_P2)

    print("SFT trainer dir:", CKPT_DIR_P2)
    print("SFT adapter dir:", ADAPTER_DIR_P2)
    print("SFT epoch checkpoints:", len(epoch_ckpts))
    print("SFT timed checkpoints:", len(timed_ckpts))
    print("SFT final adapter exists:", final_adapter_exists)

    if epoch_ckpts:
        return latest_path(epoch_ckpts, lambda p: int(p.rsplit("-", 1)[-1]))
    if final_adapter_exists:
        return ADAPTER_DIR_P2
    if timed_ckpts:
        return latest_path(timed_ckpts, lambda p: int(p.rsplit("_", 1)[-1]))
    raise FileNotFoundError("No saved SFT adapter/checkpoint found.")


def resolve_dpo_load_path():
    timed_ckpts = _glob.glob(f"{ADAPTER_DIR_DPO}/timed_ckpt_step_*")
    epoch_ckpts = _glob.glob(f"{CKPT_DIR_DPO}/checkpoint-*")
    final_adapter_exists = has_adapter(ADAPTER_DIR_DPO)

    print("DPO adapter dir:", ADAPTER_DIR_DPO)
    print("DPO trainer dir:", CKPT_DIR_DPO)
    print("DPO timed checkpoints:", len(timed_ckpts))
    print("DPO epoch checkpoints:", len(epoch_ckpts))
    print("DPO final adapter exists:", final_adapter_exists)

    if timed_ckpts:
        return latest_path(timed_ckpts, lambda p: int(p.rsplit("_", 1)[-1]))
    if final_adapter_exists:
        return ADAPTER_DIR_DPO
    if epoch_ckpts:
        return latest_path(epoch_ckpts, lambda p: int(p.rsplit("-", 1)[-1]))
    raise FileNotFoundError("No saved DPO adapter/checkpoint found.")


In [ ]:
def _parse_numeric(val_str):
    return float(val_str) if "." in str(val_str) else int(val_str)


def clean_scalar(value):
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass

    if isinstance(value, bool):
        return bool(value)

    if isinstance(value, Integral):
        return int(value)

    if isinstance(value, Real):
        return round(float(value), 4)

    return value


def _quote_string(value):
    escaped = str(value).replace("\\", "\\\\").replace("'", "\\'")
    return f"'{escaped}'"


def _format_dsl_value(value):
    if isinstance(value, str):
        return _quote_string(value)
    if isinstance(value, bool):
        return "1" if value else "0"
    return str(value)


def _split_args(arg_text):
    args = []
    cur = ""
    quote = None
    depth = 0

    for ch in arg_text:
        if quote:
            cur += ch
            if ch == quote:
                quote = None
        else:
            if ch in ["'", '"']:
                quote = ch
                cur += ch
            elif ch in "([{":
                depth += 1
                cur += ch
            elif ch in ")]}":
                depth -= 1
                cur += ch
            elif ch == "," and depth == 0:
                if cur.strip():
                    args.append(cur.strip())
                cur = ""
            else:
                cur += ch

    if cur.strip():
        args.append(cur.strip())

    return args


def _parse_arg_value(value):
    value = value.strip()

    if len(value) >= 2 and value[0] == value[-1] and value[0] in ["'", '"']:
        return value[1:-1]

    if re.fullmatch(r"-?\d+", value):
        return int(value)

    if re.fullmatch(r"-?\d+\.\d+", value):
        return float(value)

    if value.lower() == "true":
        return True

    if value.lower() == "false":
        return False

    return value


def parse_tool_invocation(invocation):
    invocation = invocation.strip()
    invocation = invocation.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    m = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)\s*\((.*)\)\s*$", invocation, re.DOTALL)

    if not m:
        return None

    name = m.group(1).strip()
    arg_text = m.group(2).strip()
    args = {}

    if arg_text:
        for part in _split_args(arg_text):
            if "=" not in part:
                continue
            key, value = part.split("=", 1)
            args[key.strip()] = _parse_arg_value(value)

    return name, args


def normalize_action_name(action_name):
    aliases = {
        "filter": "filter_data",
        "filter_rows": "filter_data",
        "groupby": "group_by",
        "group": "group_by",
        "sum": "aggregate_sum",
        "mean": "aggregate_mean",
        "average": "aggregate_mean",
        "count": "aggregate_count",
        "sort": "sort_by",
        "top": "top_k",
        "topk": "top_k",
    }
    return aliases.get(action_name.strip(), action_name.strip())


def normalize_action_input(action_name, action_input):
    action_name = normalize_action_name(action_name)
    action_input = dict(action_input)

    if action_name == "filter_data":
        if "col" in action_input and "column" not in action_input:
            action_input["column"] = action_input["col"]
        if "val" in action_input and "value" not in action_input:
            action_input["value"] = action_input["val"]
        if "equals" in action_input and "value" not in action_input:
            action_input["value"] = action_input["equals"]

        return {
            "column": str(action_input["column"]),
            "value": action_input["value"],
        }

    if action_name == "group_by":
        if "by" in action_input and "column" not in action_input:
            action_input["column"] = action_input["by"]

        return {
            "column": str(action_input["column"]),
        }

    if action_name in {"aggregate_sum", "aggregate_mean", "aggregate_count"}:
        if "field" in action_input and "column" not in action_input:
            action_input["column"] = action_input["field"]

        return {
            "column": str(action_input["column"]),
        }

    if action_name == "sort_by":
        if "field" in action_input and "column" not in action_input:
            action_input["column"] = action_input["field"]

        if "ascending" in action_input and "order" not in action_input:
            action_input["order"] = "asc" if action_input["ascending"] else "desc"

        order = str(action_input.get("order", "desc")).lower()
        if order not in {"asc", "desc"}:
            order = "desc"

        return {
            "column": str(action_input["column"]),
            "order": order,
        }

    if action_name == "top_k":
        if "top" in action_input and "k" not in action_input:
            action_input["k"] = action_input["top"]
        if "n" in action_input and "k" not in action_input:
            action_input["k"] = action_input["n"]

        return {
            "k": int(action_input["k"]),
        }

    raise ValueError(f"Unknown action: {action_name}")


def action_input_to_action_string(action_name, action_input):
    action_name = normalize_action_name(action_name)
    action_input = normalize_action_input(action_name, action_input)

    if action_name == "filter_data":
        return (
            f"filter_data(column={_quote_string(action_input['column'])}, "
            f"value={_format_dsl_value(action_input['value'])})"
        )

    if action_name == "group_by":
        return f"group_by(column={_quote_string(action_input['column'])})"

    if action_name in {"aggregate_sum", "aggregate_mean", "aggregate_count"}:
        return f"{action_name}(column={_quote_string(action_input['column'])})"

    if action_name == "sort_by":
        return (
            f"sort_by(column={_quote_string(action_input['column'])}, "
            f"order={_quote_string(action_input['order'])})"
        )

    if action_name == "top_k":
        return f"top_k(k={int(action_input['k'])})"

    raise ValueError(f"Unknown action name: {action_name}")


def parse_agent_action(action_str):
    parsed = parse_tool_invocation(action_str)
    if parsed is None:
        return None

    action_name, action_input = parsed
    action_name = normalize_action_name(action_name)

    try:
        action_input = normalize_action_input(action_name, action_input)
    except Exception:
        return None

    if action_name == "filter_data":
        return {
            "tool": "filter",
            "args": {
                "column": action_input["column"],
                "op": "==",
                "value": action_input["value"],
            },
        }

    if action_name == "group_by":
        return {
            "tool": "groupby",
            "args": {
                "column": action_input["column"],
            },
        }

    if action_name == "aggregate_sum":
        return {
            "tool": "aggregate",
            "args": {
                "column": action_input["column"],
                "agg": "sum",
            },
        }

    if action_name == "aggregate_mean":
        return {
            "tool": "aggregate",
            "args": {
                "column": action_input["column"],
                "agg": "mean",
            },
        }

    if action_name == "aggregate_count":
        return {
            "tool": "aggregate",
            "args": {
                "column": action_input["column"],
                "agg": "count",
            },
        }

    if action_name == "sort_by":
        return {
            "tool": "sort",
            "args": {
                "column": action_input["column"],
                "ascending": action_input["order"] != "desc",
            },
        }

    if action_name == "top_k":
        return {
            "tool": "topk",
            "args": {
                "k": int(action_input["k"]),
            },
        }

    return None


def execute_action_sequence(actions, source_df):
    parsed = []

    for action in actions:
        parsed_action = parse_agent_action(action)

        if parsed_action is None:
            raise ValueError(f"Could not parse action: {action}")

        parsed.append(parsed_action)

    return ToolExecutor(source_df.copy()).execute(parsed)


def result_to_python(actions, result_df):
    if result_df is None:
        return None

    if hasattr(result_df, "empty") and result_df.empty:
        return None

    if hasattr(result_df, "groups") and not isinstance(result_df, pd.DataFrame):
        return None

    action_names = [action.split("(", 1)[0] for action in actions]
    has_groupby = any(name == "group_by" for name in action_names)
    has_sort_or_topk = any(name in {"sort_by", "top_k"} for name in action_names)

    if getattr(result_df, "shape", None) == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])

    if isinstance(result_df, pd.DataFrame):
        if result_df.shape[1] == 2 and has_groupby and not has_sort_or_topk:
            key_col, value_col = result_df.columns
            return {
                str(row[key_col]): clean_scalar(row[value_col])
                for _, row in result_df.iterrows()
            }

        return [
            {str(key): clean_scalar(value) for key, value in row.items()}
            for row in result_df.to_dict(orient="records")
        ]

    return clean_scalar(result_df)


def compute_answer(actions, source_df):
    if not actions:
        return None

    try:
        result = execute_action_sequence(actions, source_df)
        return result_to_python(actions, result)
    except Exception:
        return None


def _action_name(action_str):
    parsed = parse_tool_invocation(action_str)
    return normalize_action_name(parsed[0]) if parsed else action_str.split("(", 1)[0]


def _has_action(actions, name):
    return any(_action_name(action) == name for action in actions)


def _has_filter(actions, column):
    for action in actions:
        parsed = parse_tool_invocation(action)
        if parsed and normalize_action_name(parsed[0]) == "filter_data":
            args = parsed[1]
            if str(args.get("column")) == column:
                return True
    return False


def _insert_before_first_aggregate(actions, action_to_insert):
    if action_to_insert in actions:
        return actions

    for i, action in enumerate(actions):
        if _action_name(action) in {"aggregate_sum", "aggregate_mean", "aggregate_count"}:
            return actions[:i] + [action_to_insert] + actions[i:]

    return actions + [action_to_insert]


def _infer_metric(question):
    q = question.lower()

    if "profit" in q:
        return "profit"

    if "cost" in q:
        return "cost"

    if "unit" in q or "units_sold" in q or "units sold" in q:
        return "units_sold"

    return "revenue"


def _infer_aggregate(question):
    q = question.lower()

    if "average" in q or "mean" in q:
        return "aggregate_mean"

    if "count" in q or "how many" in q or "number of" in q:
        return "aggregate_count"

    return "aggregate_sum"


def _infer_group_column(question):
    q = question.lower()

    checks = [
        ("city", ["by city", "which city", "cities"]),
        ("region", ["by region", "which region", "regions"]),
        ("product", ["by product", "which product", "products"]),
        ("category", ["by category", "which category", "categories"]),
        ("year", ["by year", "which year", "years"]),
        ("month", ["by month", "which month", "months"]),
    ]

    for col, phrases in checks:
        if any(phrase in q for phrase in phrases):
            return col

    return None


def _infer_top_k(question):
    q = question.lower()

    m = re.search(r"\btop\s+(\d+)\b", q)
    if m:
        return int(m.group(1))

    m = re.search(r"\bbottom\s+(\d+)\b", q)
    if m:
        return int(m.group(1))

    if any(word in q for word in ["highest", "maximum", "max", "most", "lowest", "minimum", "min", "least"]):
        return 1

    return None


def _question_mentions_value(question, column, value):
    q = question.lower()
    col = str(column).lower()
    v = str(value).lower()
    escaped = re.escape(v)

    if len(v) == 1:
        patterns = [
            rf"\b{col}\b\s*(?:is|=|:|as|for|of)?\s*{escaped}\b",
            rf"\b{escaped}\b\s+{col}\b",
        ]
        return any(re.search(pattern, q) for pattern in patterns)

    return re.search(rf"\b{escaped}\b", q) is not None


def repair_actions(question, actions, source_df=None):
    q = question.lower()
    fixed = list(actions)

    years = re.findall(r"\b(20\d{2})\b", q)
    for year in years:
        if not _has_filter(fixed, "year"):
            fixed.insert(0, f"filter_data(column='year', value={int(year)})")

    if source_df is not None:
        for col in ["city", "region", "product", "category"]:
            if col in source_df.columns and not _has_filter(fixed, col):
                values = sorted({str(v) for v in source_df[col].dropna().unique()}, key=len, reverse=True)
                for value in values:
                    if _question_mentions_value(question, col, value):
                        fixed.insert(0, f"filter_data(column='{col}', value='{value}')")
                        break

    metric = _infer_metric(question)
    agg = _infer_aggregate(question)
    group_col = _infer_group_column(question)
    top_k = _infer_top_k(question)

    asks_numeric = any(
        phrase in q
        for phrase in [
            "total",
            "sum",
            "average",
            "mean",
            "count",
            "how many",
            "number of",
            "highest",
            "maximum",
            "max",
            "top",
            "most",
            "lowest",
            "minimum",
            "min",
            "bottom",
            "least",
        ]
    )

    if group_col and not _has_action(fixed, "group_by"):
        fixed = _insert_before_first_aggregate(fixed, f"group_by(column='{group_col}')")

    if asks_numeric and not any(_has_action(fixed, name) for name in ["aggregate_sum", "aggregate_mean", "aggregate_count"]):
        fixed.append(f"{agg}(column='{metric}')")

    need_sort = any(word in q for word in ["highest", "maximum", "max", "top", "most", "lowest", "minimum", "min", "bottom", "least"])
    if need_sort and not _has_action(fixed, "sort_by"):
        order = "asc" if any(word in q for word in ["lowest", "minimum", "min", "bottom", "least"]) else "desc"
        fixed.append(f"sort_by(column='{metric}', order='{order}')")

    if top_k is not None and not _has_action(fixed, "top_k"):
        fixed.append(f"top_k(k={top_k})")

    return fixed


def format_observation(result_df, max_chars=700):
    if result_df is None:
        return "No data returned."

    if hasattr(result_df, "groups") and not isinstance(result_df, pd.DataFrame):
        group_names = list(result_df.groups.keys())
        sample_groups = [str(group) for group in group_names[:10]]
        return f"Grouped result with {len(group_names)} groups. Sample groups: {sample_groups}"

    if hasattr(result_df, "empty") and result_df.empty:
        return "Empty result."

    if getattr(result_df, "shape", None) == (1, 1):
        return str(clean_scalar(result_df.iloc[0, 0]))

    if isinstance(result_df, pd.DataFrame):
        text = f"DataFrame shape={result_df.shape}\n"
        text += result_df.head(8).to_string(index=False)
    else:
        text = str(result_df)

    if len(text) > max_chars:
        return text[:max_chars] + "..."

    return text


def sanitize_json(text):
    text = str(text).strip()
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    text = re.sub(r",\s*([}\]])", r"\1", text)
    return text.strip()


def parse_json_like(text):
    cleaned = sanitize_json(text)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    try:
        return ast.literal_eval(cleaned)
    except Exception:
        pass

    match = re.search(r"\{.*\}|\[.*\]", cleaned, re.DOTALL)
    if match:
        chunk = sanitize_json(match.group())

        try:
            return json.loads(chunk)
        except json.JSONDecodeError:
            pass

        try:
            return ast.literal_eval(chunk)
        except Exception:
            pass

    raise ValueError(f"Could not parse structured input: {text}")


def parse_action_input_flexible(action_name, text):
    raw = sanitize_json(text)

    if not raw:
        raise ValueError("Empty Action Input.")

    try:
        parsed = parse_json_like(raw)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    parsed_invocation = parse_tool_invocation(f"{action_name}({raw})")
    if parsed_invocation is not None:
        return parsed_invocation[1]

    raise ValueError(f"Could not parse Action Input for {action_name}: {raw}")


def normalize_react_format(text):
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("Final answer:", "Final Answer:")
    text = re.sub(
        r"\s*(Thought:|Action:|Action Input:|Observation:|Final Answer:)",
        lambda m: "\n" + m.group(1),
        text,
    )
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def strip_hallucinated_observation(text):
    if "Observation:" in text:
        return text.split("Observation:", 1)[0].strip()
    return text.strip()


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]

    return clean_scalar(obj)


class ObservationStopCriteria(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length=0):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        generated_ids = input_ids[0][self.prompt_length:]
        if generated_ids.numel() == 0:
            return False
        tail = self.tokenizer.decode(generated_ids[-80:], skip_special_tokens=True)
        return "Observation:" in tail


REACT_SYSTEM_PROMPT = """You are a data analysis ReAct agent for a sales dataset.

Available tools:
[
  {"name": "filter_data", "description": "Filter rows where a column equals a value", "parameters": {"column": "str", "value": "str or number"}},
  {"name": "group_by", "description": "Group rows by a column", "parameters": {"column": "str"}},
  {"name": "aggregate_sum", "description": "Sum a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_mean", "description": "Average a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_count", "description": "Count rows", "parameters": {"column": "str"}},
  {"name": "sort_by", "description": "Sort rows by a column", "parameters": {"column": "str", "order": "asc or desc"}},
  {"name": "top_k", "description": "Select top k rows", "parameters": {"k": "int"}}
]

Dataset schema:
date, year, month, city, region, product, category, revenue, units_sold, cost, profit

Use exactly this format:

Thought: reason about the next step
Action: tool_name
Action Input: {"arg": "value"}

After Action Input, stop. Do not generate Observation. The Python executor will provide Observation.

When enough information is available, use:

Thought: I now know the answer.
Final Answer: final answer only

Rules:
- For total revenue, total sales, or sum revenue, filter first if needed, then use aggregate_sum on revenue.
- For average revenue or mean revenue, filter/group first if needed, then use aggregate_mean on revenue.
- For total profit, use aggregate_sum on profit.
- For average profit, use aggregate_mean on profit.
- For total units sold, use aggregate_sum on units_sold.
- For by city, by region, by product, by category, by year, or by month, use group_by before aggregation.
- For highest, maximum, top, or most, aggregate first if needed, then sort_by with order desc, then top_k.
- For lowest, minimum, bottom, or least, aggregate first if needed, then sort_by with order asc, then top_k.
- Do not stop after filter_data unless the user only asks to show filtered rows.
- Do not invent tool outputs.
- Do not write Observation yourself.
- After reading an Observation, continue with the next Thought and Action if more computation is needed.
- After the final Observation gives enough information, output Thought and Final Answer.
"""

REACT_PROMPT_TEMPLATE = """### System
{system}

### User Query
{question}

### Agent Scratchpad
{scratchpad}"""


class ReActAgent:
    def __init__(
        self,
        model,
        tokenizer,
        source_df,
        max_steps=5,
        max_obs_chars=700,
        max_consecutive_errors=2,
        max_new_tokens=256,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.source_df = source_df
        self.max_steps = max_steps
        self.max_obs_chars = max_obs_chars
        self.max_consecutive_errors = max_consecutive_errors
        self.max_new_tokens = max_new_tokens

    def _append_scratchpad(self, scratchpad, block):
        block = normalize_react_format(block)

        if not block:
            return scratchpad

        if scratchpad and not scratchpad.endswith("\n"):
            scratchpad += "\n"

        return scratchpad + block + "\n"

    def run(self, question):
        self.model.eval()
        self.model.config.use_cache = True

        scratchpad = ""
        consecutive_errors = 0
        all_actions = []

        for step in range(self.max_steps):
            prompt = REACT_PROMPT_TEMPLATE.format(
                system=REACT_SYSTEM_PROMPT,
                question=question,
                scratchpad=scratchpad,
            )

            output_text = normalize_react_format(self._generate(prompt))
            output_text = strip_hallucinated_observation(output_text)

            if output_text:
                scratchpad = self._append_scratchpad(scratchpad, output_text)

            if "Final Answer:" in output_text:
                fixed_actions = repair_actions(question, all_actions, self.source_df)
                computed_answer = compute_answer(fixed_actions, self.source_df)

                if computed_answer is not None:
                    return {
                        "actions": fixed_actions,
                        "answer": make_json_safe(computed_answer),
                        "steps": step + 1,
                        "scratchpad": scratchpad,
                    }

                answer_text = output_text.split("Final Answer:", 1)[-1].strip()

                return {
                    "actions": all_actions,
                    "answer": make_json_safe(self._parse_final_answer(answer_text)),
                    "steps": step + 1,
                    "scratchpad": scratchpad,
                }

            try:
                action_name, action_input = self._parse_action(output_text)
                action_str = action_input_to_action_string(action_name, action_input)

                if parse_agent_action(action_str) is None:
                    raise ValueError(f"Unknown action: {action_str}")

                if all_actions and action_str == all_actions[-1]:
                    raise ValueError("Repeated same action. Choose the next required tool instead of repeating.")

                trial_actions = all_actions + [action_str]
                result = execute_action_sequence(trial_actions, self.source_df)
                all_actions = trial_actions
                obs = format_observation(result, max_chars=self.max_obs_chars)
                consecutive_errors = 0

            except Exception as exc:
                consecutive_errors += 1
                obs = f"ERROR - {type(exc).__name__}: {exc}"

                if consecutive_errors >= self.max_consecutive_errors:
                    fixed_actions = repair_actions(question, all_actions, self.source_df)
                    fallback_answer = compute_answer(fixed_actions, self.source_df)
                    scratchpad = self._append_scratchpad(scratchpad, f"Observation: {obs}")

                    return {
                        "actions": fixed_actions,
                        "answer": make_json_safe(fallback_answer),
                        "error": f"Max consecutive errors ({self.max_consecutive_errors}) reached.",
                        "steps": step + 1,
                        "scratchpad": scratchpad,
                    }

            scratchpad = self._append_scratchpad(scratchpad, f"Observation: {obs}")

        fixed_actions = repair_actions(question, all_actions, self.source_df)
        fallback_answer = compute_answer(fixed_actions, self.source_df)

        return {
            "actions": fixed_actions,
            "answer": make_json_safe(fallback_answer),
            "error": f"Max steps ({self.max_steps}) reached without Final Answer.",
            "steps": self.max_steps,
            "scratchpad": scratchpad,
        }

    def run_json(self, question):
        result = self.run(question)
        final = {
            "actions": result.get("actions", []),
            "answer": result.get("answer"),
        }
        return json.dumps(make_json_safe(final), ensure_ascii=False)

    def _generate(self, prompt):
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH_P2,
        ).to(self.model.device)

        prompt_length = inputs["input_ids"].shape[1]
        stop_criteria = StoppingCriteriaList([ObservationStopCriteria(self.tokenizer, prompt_length)])

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                stopping_criteria=stop_criteria,
            )

        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def _parse_action(self, text):
        text = normalize_react_format(text)

        action_line = re.search(r"Action:\s*(.*?)(?:\n|$)", text, re.DOTALL)
        input_match = re.search(
            r"Action Input:\s*(.*?)(?:\n(?:Observation:|Thought:|Action:|Final Answer:)|$)",
            text,
            re.DOTALL,
        )

        if not action_line:
            raise ValueError("Could not parse Action line.")

        raw_action = action_line.group(1).strip()

        direct_invocation = parse_tool_invocation(raw_action)
        if direct_invocation is not None:
            action_name, action_input = direct_invocation
            return normalize_action_name(action_name), action_input

        action_name = normalize_action_name(raw_action)

        if not input_match:
            raise ValueError("Could not parse Action Input line.")

        raw_input = input_match.group(1).strip()
        action_input = parse_action_input_flexible(action_name, raw_input)

        if not isinstance(action_input, dict):
            raise ValueError("Action Input must decode to a dictionary.")

        return action_name, action_input

    def _parse_final_answer(self, text):
        text = sanitize_json(text)

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        try:
            return ast.literal_eval(text)
        except Exception:
            pass

        match = re.search(r"\{.*\}|\[.*\]", text, re.DOTALL)
        if match:
            chunk = sanitize_json(match.group())

            try:
                return json.loads(chunk)
            except json.JSONDecodeError:
                pass

            try:
                return ast.literal_eval(chunk)
            except Exception:
                pass

        num_match = re.search(r"-?\d+(?:\.\d+)?", text)
        if num_match:
            val = num_match.group()
            return float(val) if "." in val else int(val)

        return text


def format_react_trace(result):
    trace = normalize_react_format(result.get("scratchpad", "")).strip()
    answer = result.get("answer")

    if isinstance(answer, (dict, list)):
        answer_text = json.dumps(make_json_safe(answer), ensure_ascii=False)
    else:
        answer_text = str(make_json_safe(answer))

    if "Final Answer:" not in trace:
        if trace:
            trace += "\n"
        trace += "Thought: I now know the answer.\n"
        trace += f"Final Answer: {answer_text}"

    return trace.strip()


def run_react_trace(agent, question, show_json=True, show_debug=False):
    result = agent.run(question)
    print(format_react_trace(result))

    if show_json:
        final_output = {
            "actions": result.get("actions", []),
            "answer": result.get("answer"),
        }
        print("\nJSON output:")
        print(json.dumps(make_json_safe(final_output), indent=2, ensure_ascii=False))

    if show_debug and result.get("error"):
        print("\ndebug error:", result.get("error"))

    return result



In [ ]:
MODEL_LABELS = {
    "sft": "ToolAlpaca + ReAct (SFT)",
    "dpo": "DPO-aligned ReAct",
}

MODEL_PATHS = {
    "sft": resolve_sft_load_path(),
    "dpo": resolve_dpo_load_path(),
}

ACTIVE_MODEL_KEY = None
ACTIVE_MODEL = None
ACTIVE_TOKENIZER = None


def unload_active_model():
    global ACTIVE_MODEL_KEY, ACTIVE_MODEL, ACTIVE_TOKENIZER

    if ACTIVE_MODEL is not None:
        try:
            del ACTIVE_MODEL
        except Exception:
            pass

    if ACTIVE_TOKENIZER is not None:
        try:
            del ACTIVE_TOKENIZER
        except Exception:
            pass

    ACTIVE_MODEL_KEY = None
    ACTIVE_MODEL = None
    ACTIVE_TOKENIZER = None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model_bundle(model_key):
    global ACTIVE_MODEL_KEY, ACTIVE_MODEL, ACTIVE_TOKENIZER

    if ACTIVE_MODEL_KEY == model_key and ACTIVE_MODEL is not None and ACTIVE_TOKENIZER is not None:
        return ACTIVE_MODEL, ACTIVE_TOKENIZER

    unload_active_model()

    load_path = MODEL_PATHS[model_key]
    print(f"Loading {MODEL_LABELS[model_key]} from: {load_path}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    tokenizer.truncation_side = "left"

    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    model = PeftModel.from_pretrained(base_model, load_path)
    model.eval()
    model.config.use_cache = True

    ACTIVE_MODEL_KEY = model_key
    ACTIVE_MODEL = model
    ACTIVE_TOKENIZER = tokenizer
    return ACTIVE_MODEL, ACTIVE_TOKENIZER


def make_agent(model_key, max_steps=MAX_STEPS_INFER):
    model, tokenizer = load_model_bundle(model_key)
    return ReActAgent(
        model,
        tokenizer,
        df,
        max_steps=max_steps,
        max_new_tokens=MAX_NEW_TOKENS_INFER,
    )


def compare_single_query(question, show_json=True, show_debug=False, max_steps=MAX_STEPS_INFER):
    outputs = {}

    for model_key in ("sft", "dpo"):
        print("=" * 100)
        print(MODEL_LABELS[model_key])
        agent = make_agent(model_key, max_steps=max_steps)
        outputs[model_key] = run_react_trace(agent, question, show_json=show_json, show_debug=show_debug)

    unload_active_model()
    return outputs


def canonical_answer(value):
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def evaluate_agent_on_subset(model_key, eval_examples):
    agent = make_agent(model_key)
    rows = []

    for idx, entry in enumerate(eval_examples, start=1):
        question = entry["query"]
        reference_actions = entry["actions"]
        reference_answer = compute_answer(reference_actions, df)

        try:
            result = agent.run(question)
        except Exception as exc:
            result = {
                "actions": [],
                "answer": None,
                "steps": [],
                "error": str(exc),
            }

        predicted_actions = result.get("actions", [])
        predicted_answer = result.get("answer")
        error_text = result.get("error")

        rows.append(
            {
                "model_key": model_key,
                "model": MODEL_LABELS[model_key],
                "idx": idx,
                "query": question,
                "reference_actions": json.dumps(reference_actions, ensure_ascii=False),
                "predicted_actions": json.dumps(predicted_actions, ensure_ascii=False),
                "reference_answer": canonical_answer(reference_answer),
                "predicted_answer": canonical_answer(predicted_answer),
                "answer_exact_match": (reference_answer is not None and predicted_answer is not None and canonical_answer(predicted_answer) == canonical_answer(reference_answer)),
                "action_exact_match": predicted_actions == reference_actions,
                "execution_success": predicted_answer is not None and not error_text,
                "nonempty_action_trace": len(predicted_actions) > 0,
                "tool_calls": len(predicted_actions),
                "steps": int(result.get("steps", 0) or 0),
                "error": error_text,
            }
        )

    details = pd.DataFrame(rows)
    summary = pd.DataFrame(
        [
            {
                "model": MODEL_LABELS[model_key],
                "examples": len(details),
                "answer_em": round(details["answer_exact_match"].mean(), 4),
                "action_em": round(details["action_exact_match"].mean(), 4),
                "execution_success_rate": round(details["execution_success"].mean(), 4),
                "nonempty_action_rate": round(details["nonempty_action_trace"].mean(), 4),
                "avg_tool_calls": round(details["tool_calls"].mean(), 2),
                "avg_steps": round(details["steps"].mean(), 2),
                "error_rate": round(details["error"].notna().mean(), 4),
            }
        ]
    )
    return summary, details


def compare_models(eval_examples):
    summary_frames = []
    detail_frames = []

    for model_key in ("sft", "dpo"):
        summary_df, detail_df = evaluate_agent_on_subset(model_key, eval_examples)
        summary_frames.append(summary_df)
        detail_frames.append(detail_df)
        unload_active_model()

    summary_df = pd.concat(summary_frames, ignore_index=True)
    detail_df = pd.concat(detail_frames, ignore_index=True)
    return summary_df, detail_df


def build_side_by_side(detail_df):
    base_cols = ["query", "reference_answer"]
    sft_cols = ["query", "predicted_answer", "answer_exact_match", "action_exact_match", "execution_success", "tool_calls", "error"]
    dpo_cols = ["query", "predicted_answer", "answer_exact_match", "action_exact_match", "execution_success", "tool_calls", "error"]

    ref_df = detail_df[detail_df["model_key"] == "sft"][base_cols].copy()
    sft_df = detail_df[detail_df["model_key"] == "sft"][sft_cols].rename(
        columns={
            "predicted_answer": "sft_answer",
            "answer_exact_match": "sft_answer_em",
            "action_exact_match": "sft_action_em",
            "execution_success": "sft_exec_success",
            "tool_calls": "sft_tool_calls",
            "error": "sft_error",
        }
    )
    dpo_df = detail_df[detail_df["model_key"] == "dpo"][dpo_cols].rename(
        columns={
            "predicted_answer": "dpo_answer",
            "answer_exact_match": "dpo_answer_em",
            "action_exact_match": "dpo_action_em",
            "execution_success": "dpo_exec_success",
            "tool_calls": "dpo_tool_calls",
            "error": "dpo_error",
        }
    )

    merged = ref_df.merge(sft_df, on="query").merge(dpo_df, on="query")
    return merged


print("Resolved load paths:")
for key, path in MODEL_PATHS.items():
    print(f"- {MODEL_LABELS[key]}: {path}")


In [ ]:
test_queries = [
    "What is the total revenue for 2022?",
    "Which city had the highest profit in 2021? Top 1",
    "What is the average revenue by city?",
]

for q in test_queries:
    print("#" * 100)
    print("Query:", q)
    compare_single_query(q, show_json=True, show_debug=True)


In [ ]:
# Handcrafted evaluation grounded in sales_data.csv.
# These queries are not sampled from agent_trajectories_2k.json.
# compare_models(eval_examples) loads SFT once for the full block, unloads it, then loads DPO once.

eval_examples = [
    {
        "query": "What is the total revenue for 2022?",
        "actions": [
            "filter_data(column='year', value=2022)",
            "aggregate_sum(column='revenue')",
        ],
    },
    {
        "query": "Which city had the highest total profit in 2021? Top 1.",
        "actions": [
            "filter_data(column='year', value=2021)",
            "group_by(column='city')",
            "aggregate_sum(column='profit')",
            "sort_by(column='profit', order='desc')",
            "top_k(k=1)",
        ],
    },
    {
        "query": "What is the average units sold by category in 2023?",
        "actions": [
            "filter_data(column='year', value=2023)",
            "group_by(column='category')",
            "aggregate_mean(column='units_sold')",
        ],
    },
    {
        "query": "What is the total cost for Electronics in 2022?",
        "actions": [
            "filter_data(column='year', value=2022)",
            "filter_data(column='category', value='Electronics')",
            "aggregate_sum(column='cost')",
        ],
    },
    {
        "query": "Which region had the lowest total revenue in 2023?",
        "actions": [
            "filter_data(column='year', value=2023)",
            "group_by(column='region')",
            "aggregate_sum(column='revenue')",
            "sort_by(column='revenue', order='asc')",
            "top_k(k=1)",
        ],
    },
    {
        "query": "Show the top 3 cities by total revenue in 2022.",
        "actions": [
            "filter_data(column='year', value=2022)",
            "group_by(column='city')",
            "aggregate_sum(column='revenue')",
            "sort_by(column='revenue', order='desc')",
            "top_k(k=3)",
        ],
    },
    {
        "query": "What is the average profit for Grocery in month 6?",
        "actions": [
            "filter_data(column='category', value='Grocery')",
            "filter_data(column='month', value=6)",
            "aggregate_mean(column='profit')",
        ],
    },
    {
        "query": "How many sales records are there for product A in the North region?",
        "actions": [
            "filter_data(column='product', value='A')",
            "filter_data(column='region', value='North')",
            "aggregate_count(column='product')",
        ],
    },
    {
        "query": "What is the total units sold by month for Delhi in 2021?",
        "actions": [
            "filter_data(column='year', value=2021)",
            "filter_data(column='city', value='Delhi')",
            "group_by(column='month')",
            "aggregate_sum(column='units_sold')",
            "sort_by(column='month', order='asc')",
        ],
    },
    {
        "query": "Which category had the highest average profit in Mumbai?",
        "actions": [
            "filter_data(column='city', value='Mumbai')",
            "group_by(column='category')",
            "aggregate_mean(column='profit')",
            "sort_by(column='profit', order='desc')",
            "top_k(k=1)",
        ],
    },
]

reference_check = [compute_answer(entry["actions"], df) for entry in eval_examples]
assert all(answer is not None for answer in reference_check), "One or more handcrafted reference trajectories are invalid."

print("Handcrafted evaluation queries:", len(eval_examples))
print("Model loading plan: load SFT once -> score all 10 queries -> unload -> load DPO once -> score all 10 queries")
summary_df, detail_df = compare_models(eval_examples)
comparison_df = build_side_by_side(detail_df)

display(summary_df)
display(
    comparison_df[
        [
            "query",
            "reference_answer",
            "sft_answer",
            "dpo_answer",
            "sft_answer_em",
            "dpo_answer_em",
            "sft_action_em",
            "dpo_action_em",
            "sft_exec_success",
            "dpo_exec_success",
        ]
    ]
)


In [ ]:
query = input("Enter query: ").strip()
if query:
    compare_single_query(query, show_json=True, show_debug=True)
